<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/04_projeto_pratico/step_01/E1_souza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Entregável 1 — Especificação e Baseline

> **Grupo:** Anderson Luiz Brandão de Souza

> **Tema/Projeto:** RecFair — sistema de recomendação de produtos com contrato utilidade + justiça

> **Data:** 07/09/2026

Este notebook contém: 
- a **especificação inicial**
- um **baseline** (uma chamada a um LLM com catálogo e vendas no prompt)
- um **golden-set congelado**
- a análise crítica das limitações.

# Glossário

Siglas e identificadores usados neste notebook e nas entregas seguintes.

| Termo | Significado |
| :--- | :--- |
| **E1, E2, E3, E4** | Entregáveis da disciplina (1 = spec + baseline; 2 = workflow/tools; 3 = multiagente; 4 = comparação final). |
| **UC** | *Use Case*: Caso de uso (UC1 consulta por categoria, UC2 categoria+marca, …). |
| **RF / RNF** | Requisito funcional / não funcional. **RF-S** = requisito do *sistema* (visão), medido já no E1 mas com sucesso não exigido nesta versão. |
| **T01 … T14** | Itens do **golden-set** (conjunto de testes congelado). T01 é o primeiro caso; os IDs não mudam entre E1–E4. |
| **e1_rate_restrict** | Taxa de aprovação em T01–T10 (família `S_*`): casos que o baseline **deve** acertar. Régua de promoção do E1. |
| **e1_rate_overall** | Taxa de aprovação em **todos** os 14 casos (inclui T11–T14, família `G_*`). Evoluções incrementais devem elevar esta nota. |
| **S_** | Família de caso sistemáticos que o E1 deve acertar: `S_exact` (ordem 7d), `S_soft` (filtro + SKUs válidos), `S_diversity` (filtro + **≥2 marcas**), `S_window` (não usar o mês inteiro), `S_abstain` (abster com o `reason` certo). |
| **G_** | Caso de **gap** do sistema (falha previsível no E1): `G_claim` (benefício sem ficha), `G_price` (orçamento sem preço praticado). |
| **units_7d** | Soma de `qt_sold` na janela 25–31/08/2026 (sem TODAY). |
| **TODAY** | Data congelada da consulta (`2026-09-01`). |
| **base_price** | Preço base de tabela no CSV/SQLite. Não é o preço praticado real (este virá de tool/API). |
| **golden-set / golden_revision** | Conjunto de casos congelado; hash SHA-256 curto do JSON dos casos. |
| **halt_reason** | Por que a execução parou: `completed`, `abstained`, `schema_invalid`. |
| **stuffing** | Colocar as tabelas inteiras no prompt (sem SQL/RAG). |
| **HITL** | Humano no loop (publica a vitrine; o agente não compra). |

# 1. Descrição do problema

Sistemas de recomendação em e-commerce tendem a maximizar clique e conversão. Na prática isso reforça **vieses de popularidade** (poucos SKUs e marcas dominam a vitrine) e **estereótipos** quando o perfil do cliente é inferido, sem contemplar adeptos de diversidade ou preferências específicas. Claims desejados (fixação, vegano, anticaspa, ocasião noturna) não são explorados ou saem sem evidência documental.

O **RecFair** trata o problema no ponto de decisão: recomendar do catálogo da loja, explicar com fonte, auditar concentração de exposição e recusar claim sem evidência. A conversão é sinal, não objetivo único. O agente **não executa compra**; apenas apresenta as melhores opções disponíveis para o cliente.

**Nesta primeira versão (V1)** o recorte é deliberadamente estreito: dada uma necessidade (pedido em linguagem natural: query em português), devolver o **Top-5 da categoria** (e da marca, se pedida) segundo as **unidades vendidas nos últimos 7 dias completos**, com abstenção quando a categoria não for determinável. Preço vigente, estoque, fichas de benefício/claim, fairness rígida e multi-turno de conversação ficam para as próximas evoluções incrementais.

# 2. Usuário-alvo e stakeholders

| Papel | Objetivo |
| :--- | :--- |
| **Usuário principal:** consumidor no chat | Descrever a necessidade em pt-BR e receber uma lista útil de produtos recomendados. |
| **Stakeholder:** compliance | Validação humana para verificar: sem PII real; sem completar perfil identitário; claims só com evidência (visão). |

# 3. Casos de uso principais

Ator = consumidor. Só UC1–UC2 são sucesso **esperado** do baseline.

| ID | Entrada | Objetivo | Saída de sucesso | E1 |
| :--- | :--- | :--- | :--- | :--- |
| UC1 | Categoria ampla (“produtos de cabelo mais vendidos”) | Top-5 da categoria na janela de 7 dias | 5 SKUs do catálogo na categoria e **≥2 marcas** (há alternativas com venda na janela) | sim |
| UC2 | Categoria + marca (“cabelos da Match”) | Top-5 filtrado | 5 SKUs da marca na categoria, ordem da janela | sim |
| UC3 | Teto de preço | Respeitar orçamento | Só itens com preço vigente ≤ teto | não (tool) |
| UC4 | Benefício / claim (“anticaspa”, “vegano”, “alta fixação”) | Adequar à ficha técnica do produto | Itens cuja ficha respalda o claim | não (RAG) |
| UC5 | Pedido sem categoria / “perfume” sem gênero / marca multi-categoria | Não chutar / alucionar | `missing_category` | sim |
| UC6 | Fora do catálogo (“protetor solar”) | Recusar | `unknown_category` | sim |


# 4. Escopo, não-objetivos e premissas

| | |
| :--- | :--- |
| **Escopo V1** | Uma consulta por execução; catálogo sintético de 40 SKUs (inspirado em produtos públicos de [O Boticário](https://www.boticario.com.br/), SKUs inventados); stuffing de `tb_catalogo` (sem preço) + `tb_vendas` diárias de agosto/2026; Top-5 ou abstenção. |
| **Não-objetivos** | Uso de RAG, SQL/tools no agente, MCP, LangGraph, memória de sessão, compra, PII real, UI web, preço/estoque no agente, documentos de claim no contexto. |
| **Premissas** | Catálogo + vendas cabem na janela do modelo; `TODAY = 2026-09-01`; janela de ranking = **2026-08-25 a 2026-08-31** (7 dias completos, **sem** TODAY); `base_price` no CSV/SQLite é o **preço base de tabela**, não o preço praticado (este virá de tool/API no E2+). Geração dos dados **sem aleatoriedade**: CSV e SQLite saem do mesmo `build_dataset()`. |

# 5. Entradas e saídas

## Entrada

Uma string em pt-BR (necessidade do cliente). Sem histórico de sessão.

## Saída

Campos futuros nascem `null` no E1 para não renegociar o schema.

```text
status: recommendation | abstention
items: 0 ou exatamente 5 × {
  sku, name, brand, category, units_7d,      # E1 preenche
  price_brl, in_stock, is_launch, is_promo,  # sempre null no E1
  explanation, citations, fairness_notes     # sempre null no E1
}
reason: missing_category | unknown_category | unknown_brand | null
halt_reason: completed | abstained | schema_invalid
```

`units_7d` é a soma de `qt_sold` na janela 25–31/08. **`in_stock` fica null**: o E1 não lê estoque.

# 6. Requisitos funcionais

| ID | Requisito | Critério (E1) | Verificação |
| :--- | :--- | :--- | :--- |
| **RF-01** | Só SKUs do catálogo | 0 SKU inventado nos casos com lista | determinística |
| **RF-02** | Filtro de categoria (e marca se pedida) | ≥80% dos casos S com lista: os 5 SKUs respeitam o filtro | determinística |
| **RF-03** | Ranking 7d + desempate `cod_sku` | Nos casos com marca especificada (T03): ordem idêntica ao gabarito pandas | determinística |
| **RF-04** | Abster sem categoria segura | T06, T08, T09 com `reason=missing_category` | determinística |
| **RF-05** | Categoria ou marca inexistente | T07 `unknown_brand`; T10 `unknown_category` | determinística |
| **RF-06** | Não usar o mês inteiro | T05: o campeão do mês (Clash, `6D2W9K`) **não** entra no Top-5 | determinística |
| **RF-07** | Diversidade quando a marca **não** foi pedida | Se a categoria tem ≥2 marcas com venda na janela, o Top-5 deve ter **n_brands ≥ 2**. Cinco itens da mesma marca é **erro** (T01, T02, T04, T05). | determinística |
| **RF-S01** | Teto de preço | T12 — esperado **falhar** (sem preço no prompt) | observação |
| **RF-S02** | Claim / benefício | T11, T13, T14 — esperado **falhar** (sem ficha) | observação |


# 7. Requisitos não funcionais e restrições

| ID | Requisito | Régua |
| :--- | :--- | :--- |
| **RNF-01** | Saída no schema Pydantic | 100% dos casos (senão `schema_invalid`) |
| **RNF-02** | Sem chaves/PII no notebook | Inspeção |
| **RNF-03** | Latência mediana do agente | &lt; 10 s |
| **RNF-04** | Caminho feliz | 1 chamada LLM; 0 tools no E1 |
| **RNF-05** | Reproducibilidade | `temperature=0`, `prompt_version=v1`, `TODAY` e CSVs congelados, `golden_revision` |
| **RNF-06** | Human-in-the-loop | O agente não finaliza compra |
| **RNF-07** | Sem memória | Cada `baseline(query)` é independente |


# 8. Recursos externos potencialmente necessários

| Recurso | Nesta entrega (E1) | Hipótese futura |
| :--- | :--- | :--- |
| Google Gemini (`model_version`) | sim | mesmo modelo nas comparações |
| CSVs `tb_catalogo` / `tb_vendas` | sim (gerados por `generate_catalog.py`) | mesmas tabelas no SQLite (E2, text-to-SQL) |
| `base_price` no catálogo | arquivo/SQLite apenas; **não** vai ao prompt | preço base de tabela; praticado = tool/API |
| Tools de preço / estoque | não no agente | E2 |
| Fichas PDF/docs por SKU | não | RAG (claims/benefícios) |
| MCP | não | se preço/estoque forem reusados por ≥2 nós |
| LangGraph | não | E2 (disciplina) |


# 9. Tipo de baseline escolhido

**Classificação**: Baseline Parcial

1. **Adequação.** A tarefa nuclear é “vitrine a partir do catálogo”. Popularidade nos últimos 7 dias é o recorte clássico de varejo **e** o viés que o RecFair quer combater. Uma chamada com stuffing é a solução mais simples que ainda lê fatos tabulares (não inventa o catálogo).
2. **Simplificado de propósito.** Sem SQL, sem preço/estoque, sem ficha, sem grafo, sem sessão. O prompt é honesto: somar a janela 25–31/08, filtrar categoria/marca, abster se a categoria for insegura, e **não** devolver cinco itens da mesma marca quando houver alternativas na janela.
3. **Comparação futura.** Mesmo golden-set (ou só cresce), mesmo schema, mesmas funções de verify. O stuffing de ~1 240 linhas diárias justifica text-to-SQL no E2: o modelo tem de filtrar a janela e somar. Erro de agregação ou cálculo pelo LLM é uma limitação medida e esperada no E1.


# 10. Critérios preliminares de sucesso

| Critério | RF/RNF | Como se mede |
| :--- | :--- | :--- |
| SKU válido | RF-01 | todo `sku` ∈ catálogo |
| Igualdade com gabarito | RF-02/03/06/07 | nos casos com lista: **SKUs e ordem idênticos** ao `gold_top_n` pandas | determinística |
| Abstenção | RF-04, RF-05 | `status` + `reason` |
| Janela 7d ≠ mês | RF-06 | gabarito de T05 não inclui Clash (`6D2W9K`) |
| Preço / claim | RF-S01, RF-S02 | casos G: falha **esperada** |
| Latência | RNF-03 | mediana s |
| Custo / tokens | RNF-04 | usage_metadata × preço indicativo |
| Schema | RNF-01 | parse Pydantic |

Duas leituras do mesmo run:

- **`e1_rate_restrict`** — só T01–T10 (`S_*`): o baseline deve performar bem; é a régua de promoção do E1.
- **`e1_rate_overall`** — todos os 14 casos: inclui gaps G (T11–T14) que o E1 não contempla; incrementos futuros devem **aumentar** esta nota sem derrubar `e1_rate_restrict`.


# 11. Configuração do ambiente


In [1]:
%pip install -q -U langchain langchain-google-genai pydantic pandas jinja2


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime
import getpass
import hashlib
import json
import logging
import os
import platform
import re
import sys
import time
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field, field_validator
from typing import Literal

# O SDK google-genai avisa que AFC via Models.generate_content não é o caminho
# recomendado. O E1 não usa tools; o aviso não indica function-calling do RecFair.
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

In [3]:
def _apply_dotenv() -> None:
    """Load KEY=VALUE from nearby .env files without overriding existing env."""
    seen: set[Path] = set()
    for root in [Path.cwd(), *Path.cwd().parents[:5]]:
        path = root / ".env"
        if path in seen or not path.is_file():
            continue
        seen.add(path)
        for raw in path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            key, val = key.strip(), val.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = val


def carregar_chave_google() -> str:
    """Resolve GOOGLE_API_KEY via env, Colab userdata ou prompt. Nunca imprime a chave."""
    if os.environ.get("GOOGLE_API_KEY"):
        return "variável de ambiente GOOGLE_API_KEY"
    if os.environ.get("GEMINI_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
        return "variável de ambiente GEMINI_API_KEY"
    try:
        from google.colab import userdata  # type: ignore

        for secret_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                value = userdata.get(secret_name)
            except Exception:
                continue
            if value:
                os.environ["GOOGLE_API_KEY"] = value
                return f"Colab userdata:{secret_name}"
    except ImportError:
        pass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")
    return "entrada manual"

_apply_dotenv()
origem_chave = carregar_chave_google()
assert os.environ.get("GOOGLE_API_KEY"), "Chave Google/Gemini não configurada."
print("Chave carregada via:", origem_chave)

Chave carregada via: variável de ambiente GOOGLE_API_KEY


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Altere aqui (ou exporte RECFAIR_MODEL_VERSION) sem editar o restante do notebook.
model_version = os.environ.get("RECFAIR_MODEL_VERSION", "gemini-3.5-flash")
TEMPERATURE = 0
PROMPT_VERSAO = "v1"
N_RECOMMEND = 5

# Gemini 3.x defaulta thinking_level=medium (~3k tokens de raciocínio por caso).
# O baseline é stuffing + schema: minimal corta latência sem mudar o contrato.
# Gemini 3.x também pode defaultar temperature=1; forçamos 0 explicitamente.
llm = ChatGoogleGenerativeAI(
    model=model_version,
    temperature=TEMPERATURE,
    thinking_level="minimal",
)

RUN_INFO = {
    "architecture_id": "baseline",
    "architecture_date": "2026-09-07",
    "modelo": model_version,
    "model_version": model_version,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "n": N_RECOMMEND,
    "today": "2026-09-01",
    "window": ["2026-08-25", "2026-08-31"],
    "thinking_level": "minimal",
}
RUN_INFO

{'architecture_id': 'baseline',
 'architecture_date': '2026-09-07',
 'modelo': 'gemini-3.5-flash',
 'model_version': 'gemini-3.5-flash',
 'temperatura': 0,
 'prompt_versao': 'v1',
 'data': '2026-09-07T20:23:29',
 'python': '3.13.8',
 'n': 5,
 'today': '2026-09-01',
 'window': ['2026-08-25', '2026-08-31'],
 'thinking_level': 'minimal'}

# 12. Dados ou entradas de exemplo

Catálogo sintético de **40 SKUs** (10 perfumaria masculina, 10 feminina, 10 corpo e banho, 10 cabelos), inspirado no catálogo público do e-commerce de O Boticário.

Artefatos de entrada:

- `tb_catalogo.csv`: `cod_sku, name_sku, brand, category, base_price`

- `tb_vendas.csv`: `date, cod_sku, qt_sold` — 01/08/2026 a 31/08/2026 (1,240 linhas)

- **`base_price` não entra no prompt.** É o preço base de tabela
    - O preço real praticado será uma tool/API simulando a operação real, onde o preço muda frequentemente.

- Público-alvo e claims **não** estão nestes CSVs; virão em documentos para RAG.

- `generate_catalog.py` não usa random: `build_dataset()` alimenta CSV e SQLite com as **mesmas** linhas   
    - `write_sqlite()` confere a igualdade


In [5]:
STEP_CANDIDATES = [
    Path.cwd(),
]
STEP_DIR = next((p for p in STEP_CANDIDATES if (p / "generate_catalog.py").exists()), Path.cwd())
if str(STEP_DIR) not in sys.path:
    sys.path.insert(0, str(STEP_DIR))

from generate_catalog import (  
    CATEGORY_BODY,
    CATEGORY_HAIR,
    CATEGORY_PERFUME_F,
    CATEGORY_PERFUME_M,
    SKU_ANTICASPA,
    SKU_CLASH,
    TODAY,
    WINDOW_END,
    WINDOW_START,
    catalog_by_sku,
    ensure_csv_files,
    generate_sales,
    gold_top_n,
    units_in_window,
)

DATA_DIR = STEP_DIR / "data"
paths = ensure_csv_files(DATA_DIR)

df_catalogo = pd.read_csv(paths["tb_catalogo"])
df_vendas = pd.read_csv(paths["tb_vendas"])
assert len(df_catalogo) == 40
assert "base_price" in df_catalogo.columns
assert "list_price" not in df_catalogo.columns
assert df_vendas["date"].min() == "2026-08-01"
assert df_vendas["date"].max() == "2026-08-31"
print("Catálogo:", df_catalogo.shape, "\nVendas:", df_vendas.shape)

Catálogo: (40, 5) 
Vendas: (1240, 3)


In [6]:
df_catalogo.head()

,cod_sku,name_sku,brand,category,base_price
0,7K2N9A,Malbec Desodorante Colônia 100ml,Malbec,perfumaria_masculina,219.9
1,Q4H8L2,Malbec Eau de Parfum 90ml,Malbec,perfumaria_masculina,279.9
2,3R1B6M,Malbec Signature Eau de Parfum 90ml,Malbec,perfumaria_masculina,379.9
3,W9C5TD,Malbec Gold Desodorante Colônia 100ml,Malbec,perfumaria_masculina,259.9
4,5J8P2X,Malbec Club Intenso Desodorante Colônia 100ml,Malbec,perfumaria_masculina,249.9


In [7]:
df_vendas.head()

,date,cod_sku,qt_sold
0,2026-08-01,7K2N9A,2
1,2026-08-01,Q4H8L2,2
2,2026-08-01,3R1B6M,2
3,2026-08-01,W9C5TD,2
4,2026-08-01,5J8P2X,2


In [8]:
sales_rows = generate_sales()
print(f"TODAY={TODAY} | janela ranking={WINDOW_START} .. {WINDOW_END} (TODAY excluído)")
print("\nGabarito de *popularidade* (units_7d DESC, sku ASC). Em cabelos isso dá 5× Match — copiar essa lista em T01 é ERRO de diversidade:\n")
for cat in (CATEGORY_PERFUME_M, CATEGORY_PERFUME_F, CATEGORY_BODY, CATEGORY_HAIR):
    print(f"=== {cat} ===")
    for i, row in enumerate(gold_top_n(sales_rows, category=cat), start=1):
        print(f"  {i}. {row['cod_sku']}  {row['units_7d']:4d}  {row['brand']:16s}  {row['name_sku']}")
    print()

tb_catalogo_prompt = df_catalogo[["cod_sku", "name_sku", "brand", "category"]].to_csv(index=False)
tb_vendas_prompt = df_vendas[["date", "cod_sku", "qt_sold"]].to_csv(index=False)
print("chars no prompt: catálogo", len(tb_catalogo_prompt), "| vendas", len(tb_vendas_prompt))

TODAY=2026-09-01 | janela ranking=2026-08-25 .. 2026-08-31 (TODAY excluído)

Gabarito de *popularidade* (units_7d DESC, sku ASC). Em cabelos isso dá 5× Match — copiar essa lista em T01 é ERRO de diversidade:

=== perfumaria_masculina ===
  1. 7K2N9A   400  Malbec            Malbec Desodorante Colônia 100ml
  2. Q4H8L2   350  Malbec            Malbec Eau de Parfum 90ml
  3. 3R1B6M   300  Malbec            Malbec Signature Eau de Parfum 90ml
  4. 5J8P2X   210  Malbec            Malbec Club Intenso Desodorante Colônia 100ml
  5. W9C5TD   210  Malbec            Malbec Gold Desodorante Colônia 100ml

=== perfumaria_feminina ===
  1. 8K2F6Q   400  Lily              Lily Eau de Parfum 75ml
  2. D1W5N9   300  Lily              Lily Gardênia Eau de Parfum 75ml
  3. 6P8H3A   250  Floratta          Floratta Red Desodorante Colônia 75ml
  4. 1M9T5B   160  Her Code          Her Code Eau de Parfum 50ml
  5. 5X3R8K   150  Coffee            Coffee Woman Seduction Desodorante Colônia 100ml

=== corpo_e

# 13. Implementação do baseline

Stuffing: uma chamada, prompt versionado `v1`, tabelas interpoladas com `.format`. Sem tools. Sem memória.

A saída usa `with_structured_output(..., method="json_schema")` — schema JSON nativo do Gemini, não function calling. O aviso de AFC do SDK (`Models.generate_content`) aparece se o logger não for silenciado; **não** significa que o RecFair V1 tenha tools.


In [9]:
class RecommendationItem(BaseModel):
    sku: str
    name: str
    brand: str
    category: str
    units_7d: int
    price_brl: float | None = None
    in_stock: bool | None = None
    is_launch: bool | None = None
    is_promo: bool | None = None
    explanation: str | None = None
    citations: list[str] | None = None
    fairness_notes: str | None = None


class RecFairOutput(BaseModel):
    status: Literal["recommendation", "abstention"]
    items: list[RecommendationItem] = Field(default_factory=list)
    reason: Literal["missing_category", "unknown_category", "unknown_brand"] | None = None
    halt_reason: Literal["completed", "abstained", "schema_invalid"]

    @field_validator("items")
    @classmethod
    def _five_or_empty(cls, items: list[RecommendationItem]) -> list[RecommendationItem]:
        if items and len(items) != N_RECOMMEND:
            raise ValueError(f"recommendation must have exactly {N_RECOMMEND} items")
        return items


RecFairOutput.model_rebuild()

In [10]:
PROMPT_TEMPLATE = """Você é o RecFair, assistente de vitrine de e-commerce.

Data da consulta (TODAY): {today}
Janela de ranking: últimos 7 dias COMPLETOS, SEM incluir TODAY: {window_start} a {window_end} (inclusive).
Some apenas qt_sold nessa janela. Não use o mês inteiro nem vendas de TODAY.

Catálogo (tb_catalogo):
{tb_catalogo}

Vendas diárias (tb_vendas):
{tb_vendas}

Tarefa: para a consulta do cliente, devolva exatamente 5 SKUs (Top-5) OU abstenha.

Regras:
1. Interprete a categoria como no máximo um de: perfumaria_masculina, perfumaria_feminina, corpo_e_banho, cabelos.
2. Se a categoria não puder ser determinada com segurança (ausente; "perfume(s)" sem gênero; marca que aparece em várias categorias sem categoria explícita), status=abstention e reason=missing_category.
3. Categoria pedida que não existe no catálogo: reason=unknown_category.
4. Marca pedida que não existe no catálogo: reason=unknown_brand.
5. Filtre pela categoria (obrigatória) e pela marca (somente se o cliente a especificou).
6. Ordene por soma de qt_sold na janela (maior primeiro). Empate: cod_sku ascendente.
7. Se o cliente NÃO especificou marca e existirem SKUs de outras marcas com vendas na janela na mesma categoria, é ERRO devolver os 5 itens da mesma marca. Inclua pelo menos duas marcas. A utilidade (vendas na janela) continua o critério de ordenação entre os candidatos.
8. Não invente SKU, nome, marca ou units_7d. units_7d deve ser a soma da janela.
9. Deixe price_brl, in_stock, is_launch, is_promo, explanation, citations e fairness_notes como null. Você não tem preço praticado, estoque, benefícios nem claims.
10. Não complete o perfil do cliente com estereótipo. Sem memória de turnos anteriores.

Consulta do cliente:
{query}
"""

In [11]:
def build_prompt(query: str) -> str:
    return PROMPT_TEMPLATE.format(
        today=TODAY.isoformat(),
        window_start=WINDOW_START.isoformat(),
        window_end=WINDOW_END.isoformat(),
        tb_catalogo=tb_catalogo_prompt,
        tb_vendas=tb_vendas_prompt,
        query=query.strip(),
    )


def _message_text(raw) -> str:
    """Flatten Gemini 3 content blocks (list of dicts) into a JSON string."""
    content = getattr(raw, "content", raw)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text") or ""))
        return "".join(parts)
    return str(content or "")


def _usage(raw) -> tuple[int | None, int | None]:
    meta = getattr(raw, "usage_metadata", None) or {}
    if not isinstance(meta, dict):
        return (
            getattr(meta, "input_tokens", None),
            getattr(meta, "output_tokens", None),
        )
    return meta.get("input_tokens"), meta.get("output_tokens")


def _parse_output(packed) -> tuple[RecFairOutput | None, object | None]:
    """Unpack with_structured_output(include_raw=True); fall back to JSON in content."""
    raw = None
    parsed = None
    if isinstance(packed, dict) and "parsed" in packed:
        raw = packed.get("raw")
        parsed = packed.get("parsed")
    elif isinstance(packed, RecFairOutput):
        parsed = packed
    if parsed is None and raw is not None:
        text = _message_text(raw)
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                parsed = RecFairOutput.model_validate_json(match.group(0))
            except Exception:
                parsed = None
    return parsed, raw


# json_schema = native Gemini structured output. method=function_calling dispara AFC
# em Models.generate_content (aviso do SDK + parse frágil).
structured_llm = llm.with_structured_output(
    RecFairOutput,
    include_raw=True,
    method="json_schema",
)

In [12]:
def baseline(entrada: str) -> tuple[RecFairOutput, dict]:
    """Uma chamada Gemini: stuffing das tabelas + consulta. Sem tools e sem sessão."""
    prompt = build_prompt(entrada)
    inicio = time.perf_counter()
    try:
        packed = structured_llm.invoke(prompt)
        latencia = time.perf_counter() - inicio
        parsed, raw = _parse_output(packed)
        tokens_in, tokens_out = _usage(raw)
        if parsed is None:
            parsed = RecFairOutput(status="abstention", reason=None, halt_reason="schema_invalid")
        metricas = {
            "latencia_s": round(latencia, 2),
            "tokens_entrada": tokens_in,
            "tokens_saida": tokens_out,
            "chamadas_llm": 1,
            "tool_calls": 0,
        }
        return parsed, metricas
    except Exception as exc:
        latencia = time.perf_counter() - inicio
        fallback = RecFairOutput(
            status="abstention",
            reason=None,
            halt_reason="schema_invalid",
        )
        metricas = {
            "latencia_s": round(latencia, 2),
            "tokens_entrada": None,
            "tokens_saida": None,
            "chamadas_llm": 1,
            "tool_calls": 0,
            "erro": f"{type(exc).__name__}: {exc}",
        }
        print(f"[baseline] {metricas['erro']}")
        return fallback, metricas


print("prompt_version:", PROMPT_VERSAO, "| chars template:", len(PROMPT_TEMPLATE))

prompt_version: v1 | chars template: 1715


# 14. Conjunto de avaliação

Escrito **antes** de tunar o prompt. Congelado: os mesmos IDs serão reexecutados nos E2–E4. Novos casos só por acréscimo.

| ID | Tipo | Entrada (resumo) | Esperado V1 |
| :--- | :--- | :--- | :--- |
| T01 | normal + **diversidade** | produtos de cabelo mais vendidos | 5 SKUs em `cabelos` e **n_brands ≥ 2** (5× Match = reprovação) |
| T02 | paráfrase | o que mais sai de cabelo nesta semana | idem T01 (diversidade) |
| T03 | composto / marca | cabelos da linha Match | ordem exata do gabarito Match (5× Match **ok**) |
| T04 | normal + diversidade | corpo e banho mais vendidos | 5 SKUs na categoria e **n_brands ≥ 2** |
| T05 | janela 7d + diversidade | colônias masculinas mais vendidas | sem Clash (`6D2W9K`); **n_brands ≥ 2** |
| T06 | informação ausente | “algo bom” / presente | `missing_category` |
| T07 | marca inexistente | shampoos da Zorblax | `unknown_brand` |
| T08 | ambíguo | perfumes mais vendidos | `missing_category` |
| T09 | ambíguo (marca multi-cat.) | mais vendidos da Malbec | `missing_category` |
| T10 | fora de sortimento | protetor solar até 50 | `unknown_category` |
| T11 | gap - claim | shampoo anticaspa | falha previsível (sem ficha) |
| T12 | gap - preço | colônia masculina até R$180 | falha previsível (sem preço) |
| T13 | gap - claim | hidratante vegano | falha previsível |
| T14 | gap - claim | perfume feminino de alta fixação / noite | falha previsível |


### Chaves de cada item em `test_cases`

| Chave | Obrigatória | Significado |
| :--- | :--- | :--- |
| `id` | sim | Identificador estável (`T01`…). Não reutilizar nem apagar. |
| `tipo` | sim | Rótulo humano (normal, paráfrase, ambíguo, G-claim, …). |
| `familia` | sim | Ramo do `verify_case` (`S_exact`, `S_soft`, `S_diversity`, `S_window`, `S_abstain`, `G_need`, `G_price`). `S_*` → `e1_rate_restrict`; `G_*` → só `e1_rate_overall` (gap previsto no E1). |
| `verificacao` | sim | Sempre `auto` neste E1 (sem rubrica humana). |
| `entrada` | sim | Query em pt-BR enviada ao baseline. |
| `category` | se houver lista | Categoria canônica esperada no filtro (`perfumaria_masculina`, …). |
| `brand` | se o cliente pediu marca | Marca a filtrar; `None` = não pediu (aí vale diversidade). |
| `require_diversity` | se `brand` é `None` e há lista | `True` = reprova vitrine com uma só marca quando há alternativas na janela. |
| `expected_reason` | em `S_abstain` | Código de abstenção (`missing_category`, `unknown_brand`, `unknown_category`). |
| `forbidden_skus` | em `S_window` | SKUs que não podem aparecer (`6D2W9K` Clash = campeão do mês, não da janela). |
| `target_sku` | em G-claim | SKU que *deveria* ser escolhido com ficha (ex. `H8Q3N1` anticaspa); só observação no E1. |


In [13]:
CATALOG = catalog_by_sku()

test_cases = [
    {
        "id": "T01",
        "tipo": "normal",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Quais os produtos de cabelo mais vendidos?",
        "category": CATEGORY_HAIR,
        "brand": None,
        "require_diversity": True,
    },
    {
        "id": "T02",
        "tipo": "paráfrase",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Me indica o que mais sai de cabelo nesta semana.",
        "category": CATEGORY_HAIR,
        "brand": None,
        "require_diversity": True,
    },
    {
        "id": "T03",
        "tipo": "composto",
        "familia": "S_exact",
        "verificacao": "auto",
        "entrada": "Quais os produtos de cabelo da Match mais vendidos?",
        "category": CATEGORY_HAIR,
        "brand": "Match",
    },
    {
        "id": "T04",
        "tipo": "normal",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Quais os produtos de corpo e banho mais vendidos?",
        "category": CATEGORY_BODY,
        "brand": None,
        "require_diversity": True,
    },
    {
        "id": "T05",
        "tipo": "janela_7d",
        "familia": "S_window",
        "verificacao": "auto",
        "entrada": "Quais as colônias masculinas mais vendidas?",
        "category": CATEGORY_PERFUME_M,
        "brand": None,
        "forbidden_skus": [SKU_CLASH],
        "require_diversity": True,
    },
    {
        "id": "T06",
        "tipo": "informação ausente",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Me recomenda algo bom para presentear.",
        "expected_reason": "missing_category",
    },
    {
        "id": "T07",
        "tipo": "informação ausente",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quero os shampoos mais vendidos da Zorblax.",
        "expected_reason": "unknown_brand",
    },
    {
        "id": "T08",
        "tipo": "ambíguo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quais os perfumes mais vendidos?",
        "expected_reason": "missing_category",
    },
    {
        "id": "T09",
        "tipo": "ambíguo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quais os mais vendidos da Malbec?",
        "expected_reason": "missing_category",
    },
    {
        "id": "T10",
        "tipo": "fora de escopo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Preciso de um protetor solar até 50 reais.",
        "expected_reason": "unknown_category",
    },
    {
        "id": "T11",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Quero um shampoo anticaspa.",
        "category": CATEGORY_HAIR,
        "brand": None,
        "target_sku": SKU_ANTICASPA,
    },
    {
        "id": "T12",
        "tipo": "G-preço",
        "familia": "G_price",
        "verificacao": "auto",
        "entrada": "Colônia masculina até 180 reais, as mais vendidas.",
        "category": CATEGORY_PERFUME_M,
        "brand": None,
    },
    {
        "id": "T13",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Me indica um hidratante corporal vegano.",
        "category": CATEGORY_BODY,
        "brand": None,
    },
    {
        "id": "T14",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Perfume feminino de alta fixação para ocasião noturna.",
        "category": CATEGORY_PERFUME_F,
        "brand": None,
    },
]

blob = json.dumps(test_cases, sort_keys=True, ensure_ascii=False).encode("utf-8")
golden_revision = hashlib.sha256(blob).hexdigest()[:16]
RUN_INFO["golden_revision"] = golden_revision
print(len(test_cases), "casos | golden_revision =", golden_revision)


14 casos | golden_revision = 3177457e725ca1b9


# 15. Implementação da verificação


In [14]:
def gold_for(caso: dict) -> list[str]:
    if caso.get("category") is None:
        return []
    rows = gold_top_n(
        sales_rows,
        category=caso["category"],
        brand=caso.get("brand"),
        n=N_RECOMMEND,
    )
    return [r["cod_sku"] for r in rows]


def verify_case(caso: dict, saida: RecFairOutput) -> dict:
    """Deterministic checks against golden-set. G cases always fail in E1."""
    gold = gold_for(caso)
    skus = [item.sku for item in saida.items]
    invented = [s for s in skus if s not in CATALOG]
    cats_ok = True
    brands_ok = True
    if saida.status == "recommendation" and caso.get("category"):
        cats_ok = all(CATALOG[s]["category"] == caso["category"] for s in skus if s in CATALOG)
        if caso.get("brand"):
            brands_ok = all(CATALOG[s]["brand"] == caso["brand"] for s in skus if s in CATALOG)
    n_brands = len({CATALOG[s]["brand"] for s in skus if s in CATALOG}) if skus else 0
    gold_brands = len({CATALOG[s]["brand"] for s in gold}) if gold else 0
    order_match = skus == gold if gold else False
    forbidden = set(caso.get("forbidden_skus") or [])
    forbidden_hit = bool(forbidden.intersection(skus))
    diversity_ok = (n_brands >= 2) if caso.get("require_diversity") else True

    familia = caso["familia"]
    lista_ok = (
        saida.status == "recommendation"
        and len(skus) == N_RECOMMEND
        and not invented
        and cats_ok
        and brands_ok
    )
    aprovado = False
    if familia == "S_exact":
        aprovado = lista_ok and order_match
    elif familia == "S_soft":
        aprovado = lista_ok and order_match
    elif familia == "S_diversity":
        aprovado = lista_ok and order_match
    elif familia == "S_window":
        aprovado = lista_ok and order_match and not forbidden_hit
    elif familia == "S_abstain":
        aprovado = saida.status == "abstention" and saida.reason == caso["expected_reason"]
    elif familia in {"G_need", "G_price"}:
        aprovado = False

    return {
        "aprovado": bool(aprovado),
        "gold": gold,
        "skus": skus,
        "invented": invented,
        "order_match": order_match,
        "n_brands": n_brands,
        "gold_n_brands": gold_brands,
        "diversity_ok": diversity_ok,
        "forbidden_hit": forbidden_hit,
        "g_target_hit": (caso.get("target_sku") in skus) if caso.get("target_sku") else None,
        "reason": saida.reason,
        "halt_reason": saida.halt_reason,
        "status": saida.status,
    }


print("verify pronto; exemplo T03 gold =", gold_for(test_cases[2]))


def is_restrict_scope(familia: str) -> bool:
    """True for S_* families (T01–T10): baseline should pass."""
    return familia.startswith("S_")


def gold_diff_label(skus: list[str], gold: list[str]) -> str:
    """Human-readable diff between model output and pandas gold."""
    if not gold:
        return "—"
    if skus == gold:
        return "igual ao gabarito"
    parts: list[str] = []
    only_out = [s for s in skus if s not in gold]
    only_gold = [s for s in gold if s not in skus]
    if only_out:
        parts.append(f"extras na saída: {only_out}")
    if only_gold:
        parts.append(f"faltam do gabarito: {only_gold}")
    if skus and gold and set(skus) == set(gold):
        parts.append("mesmos SKUs, ordem diferente")
    return "; ".join(parts) if parts else "lista diferente"


def format_baseline_col(chk: dict) -> str:
    if chk.get("status") == "abstention":
        return f"abstention · {chk.get('reason')}"
    skus = chk.get("skus") or []
    return " → ".join(skus) if skus else "—"


def format_gabarito_col(caso: dict, chk: dict) -> str:
    if caso["familia"] == "S_abstain":
        return f"abstention · {caso['expected_reason']}"
    gold = chk.get("gold") or []
    base = " → ".join(gold) if gold else "—"
    if caso["familia"] == "G_need" and caso.get("target_sku"):
        return f"{base} (+ claim {caso['target_sku']})"
    if caso["familia"] == "G_price":
        return f"{base} (+ filtro preço)"
    return base


def final_status(aprovado: bool, restrict: bool) -> str:
    if aprovado:
        return "sucesso"
    return "erro" if restrict else "erro*"


def escopo_label(restrict: bool) -> str:
    return "restrito" if restrict else "global"


def diagnose_failure(caso: dict, chk: dict) -> str:
    """Detailed rejection reason for report tables."""
    if chk["aprovado"]:
        return ""
    familia = caso["familia"]
    reasons: list[str] = []
    gold = chk.get("gold") or []

    if familia.startswith("G_"):
        if gold and not chk.get("order_match") and chk.get("status") == "recommendation":
            reasons.append(f"gabarito ≠ saída: {gold_diff_label(chk.get('skus', []), gold)}")
        if familia == "G_need":
            tgt = caso.get("target_sku")
            if tgt and not chk.get("g_target_hit"):
                reasons.append(f"gap E1: claim — gabarito exige {tgt}")
            else:
                reasons.append("gap E1: claim/necessidade sem ficha no contexto")
        elif familia == "G_price":
            reasons.append("gap E1: orçamento sem preço praticado no contexto")
        if chk.get("invented"):
            reasons.append(f"RF-01: SKUs inventados {chk['invented']}")
        return " | ".join(reasons) if reasons else "falha prevista no E1 (fora do escopo do baseline)"

    if chk.get("invented"):
        reasons.append(f"RF-01: SKUs fora do catálogo {chk['invented']}")
    if familia == "S_abstain":
        exp = caso.get("expected_reason")
        got = chk.get("reason")
        if chk.get("status") != "abstention":
            reasons.append(f"RF-04/05: deveria abster, obteve status={chk.get('status')}")
        elif got != exp:
            reasons.append(f"RF-04/05: reason={got}, esperado {exp}")
        return " | ".join(reasons) if reasons else "abstenção incorreta"
    if chk.get("status") != "recommendation":
        reasons.append(f"status={chk.get('status')} (esperado recommendation)")
    if len(chk.get("skus", [])) != N_RECOMMEND:
        reasons.append(f"lista com {len(chk.get('skus', []))} SKUs (esperado {N_RECOMMEND})")
    if gold and not chk.get("order_match"):
        reasons.append(f"gabarito ≠ saída: {gold_diff_label(chk.get('skus', []), gold)}")
    if familia == "S_window" and chk.get("forbidden_hit"):
        reasons.append(f"RF-06: SKU proibido presente — {caso.get('forbidden_skus')}")
    return " | ".join(reasons) if reasons else "critério da família não satisfeito"


def _status_badge(status: str) -> str:
    """Pill badge with explicit dark-on-light contrast."""
    palette = {
        "sucesso": ("#d1e7dd", "#0a3622", "#a3cfbb"),
        "erro": ("#f8d7da", "#58151c", "#f1aeb5"),
        "erro*": ("#fff3cd", "#664d03", "#ffe69c"),
    }
    bg, fg, border = palette.get(status, ("#e9ecef", "#343a40", "#ced4da"))
    return (
        f'<span style="display:inline-block;padding:3px 10px;border-radius:999px;'
        f"background:{bg};color:{fg};border:1px solid {border};"
        f'font-weight:600;font-size:12px;letter-spacing:0.02em;">{status}</span>'
    )


def _fmt_cell(col: str, text: str, status: str) -> str:
    """Format one table cell; SKUs monospace, errors highlighted."""
    base = "padding:10px 12px;border-bottom:1px solid #e9ecef;color:#212529;vertical-align:top;"
    if col == "status_final":
        return f'<td style="{base}">{_status_badge(status)}</td>'
    if col in {"baseline", "gabarito"}:
        return (
            f'<td style="{base}">'
            f'<code style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;'
            f"font-size:12px;color:#212529;background:#f8f9fa;padding:4px 6px;"
            f'border-radius:4px;display:inline-block;max-width:420px;word-break:break-all;">'
            f"{text}</code></td>"
        )
    if col == "motivo_erro" and text not in {"—", ""}:
        return (
            f'<td style="{base}background:#fff5f5;color:#842029;font-size:12px;">{text}</td>'
        )
    if col == "diff" and text not in {"—", "", "igual ao gabarito"}:
        return (
            f'<td style="{base}background:#fff8e6;color:#664d03;font-size:12px;">{text}</td>'
        )
    return f'<td style="{base}">{text}</td>'


def _render_table_body(df: pd.DataFrame, status_col: str = "status_final") -> str:
    """Table + legend (inner content of the comparison card)."""
    accent = {"sucesso": "#198754", "erro": "#dc3545", "erro*": "#fd7e14"}
    parts = [
        '<table style="border-collapse:collapse;width:100%;min-width:720px;background:#ffffff;">',
        '<thead><tr>',
    ]
    for col in df.columns:
        label = str(col).replace("_", " ")
        parts.append(
            f'<th style="padding:10px 12px;background:#374151;color:#f9fafb;text-align:left;'
            f'font-size:11px;text-transform:uppercase;letter-spacing:0.06em;border-bottom:1px solid #4b5563;">'
            f"{label}</th>"
        )
    parts.append("</tr></thead><tbody>")
    for i, row in df.iterrows():
        st = str(row.get(status_col, ""))
        stripe = "#f9fafb" if i % 2 else "#ffffff"
        border_color = accent.get(st, "#9ca3af")
        parts.append(f'<tr style="background:{stripe};">')
        for j, col in enumerate(df.columns):
            val = row[col]
            if val is None or (isinstance(val, float) and pd.isna(val)):
                text = "—"
            else:
                text = str(val)
            if j == 0:
                parts.append(
                    f'<td style="padding:10px 12px;border-bottom:1px solid #e5e7eb;border-left:4px solid {border_color};'
                    f'color:#111827;font-weight:700;font-variant-numeric:tabular-nums;vertical-align:top;">{text}</td>'
                )
            else:
                parts.append(_fmt_cell(col, text, st))
        parts.append("</tr>")
    parts.append("</tbody></table>")
    parts.append(
        '<div style="display:flex;flex-wrap:wrap;gap:14px;padding:10px 14px 12px;font-size:12px;'
        'color:#374151;background:#f9fafb;border-top:1px solid #e5e7eb;">'
        '<span style="color:#111827;"><span style="display:inline-block;width:10px;height:10px;border-radius:2px;'
        'background:#198754;margin-right:6px;"></span>sucesso</span>'
        '<span style="color:#111827;"><span style="display:inline-block;width:10px;height:10px;border-radius:2px;'
        'background:#dc3545;margin-right:6px;"></span>erro (restrito)</span>'
        '<span style="color:#111827;"><span style="display:inline-block;width:10px;height:10px;border-radius:2px;'
        'background:#fd7e14;margin-right:6px;"></span>erro* (gap E1)</span>'
        "</div>"
    )
    return "".join(parts)


# Shared panel chrome: opaque card readable in Jupyter light AND dark themes.
_PANEL = (
    "color-scheme:only light;forced-color-adjust:none;"
    "background:#ffffff;color:#111827;"
    "border:1px solid #9ca3af;border-radius:12px;"
    "overflow:hidden;margin:0.75rem 0;"
    "box-shadow:0 2px 10px rgba(0,0,0,.18);"
    "font-family:system-ui,-apple-system,Segoe UI,sans-serif;"
)
_PANEL_HDR = (
    "background:#1e293b;color:#f8fafc;padding:14px 18px;"
    "border-bottom:2px solid #334155;"
)


def render_comparison_report(df: pd.DataFrame, status_col: str = "status_final") -> str:
    """Full comparison card: dark title bar + light table body."""
    return (
        f'<div style="{_PANEL}">'
        f'<div style="{_PANEL_HDR}">'
        '<div style="font-size:1.08rem;font-weight:700;color:#ffffff;margin:0;'
        'letter-spacing:-0.01em;">Baseline × gabarito (golden-set)</div>'
        '<div style="font-size:12px;color:#e2e8f0;margin-top:5px;line-height:1.4;">'
        "Comparação determinística: mesmos SKUs e mesma ordem = sucesso.</div>"
        "</div>"
        f'<div style="overflow-x:auto;background:#ffffff;">{_render_table_body(df, status_col)}</div>'
        "</div>"
    )


def render_metrics_panel(resumo: dict, restrict_ok: int, restrict_n: int, overall_ok: int, overall_n: int) -> str:
    """Metrics card with the same dark header pattern."""
    rb = lambda ok, n: f"{ok}/{n} ({100*ok/n:.1f}%)" if n else "—"
    return (
        f'<div style="{_PANEL}max-width:640px;">'
        f'<div style="{_PANEL_HDR}">'
        '<div style="font-size:1.08rem;font-weight:700;color:#ffffff;margin:0;">'
        "Métricas consolidadas</div>"
        '<div style="font-size:12px;color:#e2e8f0;margin-top:5px;">'
        "Run do baseline V1 · seção 17</div>"
        "</div>"
        '<table style="border-collapse:collapse;width:100%;background:#ffffff;color:#111827;">'
        f'<tr><td style="padding:11px 16px;border-bottom:1px solid #e5e7eb;background:#f9fafb;color:#111827;">'
        "<b>e1_rate_restrict</b><br>"
        '<small style="color:#4b5563;">T01–T10 · baseline deve acertar</small></td>'
        f'<td style="padding:11px 16px;border-bottom:1px solid #e5e7eb;font-size:1.15em;color:#111827;">'
        f"<b>{rb(restrict_ok, restrict_n)}</b></td></tr>"
        f'<tr><td style="padding:11px 16px;border-bottom:1px solid #e5e7eb;background:#f9fafb;color:#111827;">'
        "<b>e1_rate_overall</b><br>"
        '<small style="color:#4b5563;">T01–T14 · inclui gaps G</small></td>'
        f'<td style="padding:11px 16px;border-bottom:1px solid #e5e7eb;font-size:1.15em;color:#111827;">'
        f"<b>{rb(overall_ok, overall_n)}</b></td></tr>"
        f'<tr><td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#4b5563;">Latência mediana</td>'
        f'<td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#111827;">'
        f"<b>{resumo['latencia_mediana_s']} s</b></td></tr>"
        f'<tr><td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#4b5563;">Latência média</td>'
        f'<td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#111827;">'
        f"<b>{resumo['latencia_media_s']} s</b></td></tr>"
        f'<tr><td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#4b5563;">Chamadas LLM</td>'
        f'<td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#111827;">'
        f"<b>{resumo['chamadas_llm']}</b></td></tr>"
        f'<tr><td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#4b5563;">Tokens entrada / saída</td>'
        f'<td style="padding:10px 16px;border-bottom:1px solid #e5e7eb;color:#111827;">'
        f"<b>{resumo['tokens_entrada']:,}</b> / <b>{resumo['tokens_saida']:,}</b></td></tr>"
        f'<tr><td style="padding:10px 16px;color:#4b5563;">Custo estimado (USD)</td>'
        f'<td style="padding:10px 16px;color:#111827;">'
        f"<b>${resumo['custo_estimado_usd']:.4f}</b></td></tr>"
        "</table></div>"
    )


verify pronto; exemplo T03 gold = ['F3P9W2', 'L6K1C8', '2Y8N4T', 'R5B7Q3', '9C4M1H']


# 16. Experimentos

Cada caso: 1 chamada, latência, tokens, `halt_reason`. Sem memória entre linhas.

Ao final, tabela **baseline × gabarito** com `status_final` (`sucesso` · `erro` · `erro*`), `tipo_teste` (`restrito` / `global`) e motivo quando houver falha.

In [15]:
try:
    from IPython.display import HTML, display
except ImportError:
    def display(x):
        print(x)

    def HTML(x):
        return x

registros = []

for caso in test_cases:
    saida, metricas = baseline(caso["entrada"])
    checagem = verify_case(caso, saida)
    restrict = is_restrict_scope(caso["familia"])
    motivo = diagnose_failure(caso, checagem)
    status = final_status(checagem["aprovado"], restrict)
    registros.append(
        {
            "id": caso["id"],
            "tipo": caso["tipo"],
            "familia": caso["familia"],
            "escopo_restrict": restrict,
            "entrada": caso["entrada"],
            "saida": saida.model_dump(),
            **checagem,
            "gold_diff": gold_diff_label(checagem["skus"], checagem["gold"]),
            "motivo": motivo,
            "status": status,
            "escopo": escopo_label(restrict),
            "baseline_fmt": format_baseline_col(checagem),
            "gabarito_fmt": format_gabarito_col(caso, checagem),
            **metricas,
        }
    )

experimentos_df = pd.DataFrame(registros)
tabela_cols = [
    "id",
    "tipo",
    "escopo",
    "status",
    "baseline_fmt",
    "gabarito_fmt",
    "gold_diff",
    "motivo",
]
tabela = experimentos_df[tabela_cols].rename(
    columns={
        "id": "caso",
        "tipo": "tipo_caso",
        "escopo": "tipo_teste",
        "status": "status_final",
        "baseline_fmt": "baseline",
        "gabarito_fmt": "gabarito",
        "gold_diff": "diff",
        "motivo": "motivo_erro",
    }
)
tabela["motivo_erro"] = tabela["motivo_erro"].replace("", "—")

display(HTML(render_comparison_report(tabela, "status_final")))

n_ok = int((experimentos_df["status"] == "sucesso").sum())
n_err = int((experimentos_df["status"] == "erro").sum())
n_err_star = int((experimentos_df["status"] == "erro*").sum())
print(
    f"{len(registros)} execuções · sucesso={n_ok} · erro={n_err} · erro*={n_err_star}"
)


caso,tipo caso,tipo teste,status final,baseline,gabarito,diff,motivo erro
T01,normal,restrito,erro,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,extras na saída: ['A8T3K5']; faltam do gabarito: ['9C4M1H'],gabarito ≠ saída: extras na saída: ['A8T3K5']; faltam do gabarito: ['9C4M1H']
T02,paráfrase,restrito,erro,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,extras na saída: ['A8T3K5']; faltam do gabarito: ['9C4M1H'],gabarito ≠ saída: extras na saída: ['A8T3K5']; faltam do gabarito: ['9C4M1H']
T03,composto,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,igual ao gabarito,—
T04,normal,restrito,erro,24A51X → K8M2Q1 → 9P3W7C → B7F4L9 → Z5C1R8,24A51X → K8M2Q1 → 9P3W7C → B7F4L9 → T2N8H4,extras na saída: ['Z5C1R8']; faltam do gabarito: ['T2N8H4'],gabarito ≠ saída: extras na saída: ['Z5C1R8']; faltam do gabarito: ['T2N8H4']
T05,janela_7d,restrito,sucesso,7K2N9A → Q4H8L2 → 3R1B6M → 5J8P2X → W9C5TD,7K2N9A → Q4H8L2 → 3R1B6M → 5J8P2X → W9C5TD,igual ao gabarito,—
T06,informação ausente,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T07,informação ausente,restrito,sucesso,abstention · unknown_brand,abstention · unknown_brand,—,—
T08,ambíguo,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T09,ambíguo,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T10,fora de escopo,restrito,sucesso,abstention · unknown_category,abstention · unknown_category,—,—


14 execuções · sucesso=7 · erro=3 · erro*=4


# 17. Resultados

Métricas consolidadas do run: `e1_rate_restrict`, `e1_rate_overall`, latência, custo, tokens e chamadas LLM.
A tabela caso a caso está na seção 16.

In [16]:
try:
    from IPython.display import HTML, display
except ImportError:
    def display(x):
        print(x)

    def HTML(x):
        return x


df = experimentos_df
restrict_df = df[df["escopo_restrict"]]
n_restrict_ok = int(restrict_df["aprovado"].sum())
n_restrict = len(restrict_df)
n_overall_ok = int(df["aprovado"].sum())
n_overall = len(df)

USD_IN = 0.10 / 1_000_000
USD_OUT = 0.40 / 1_000_000
tok_in = pd.to_numeric(df["tokens_entrada"], errors="coerce").fillna(0).sum()
tok_out = pd.to_numeric(df["tokens_saida"], errors="coerce").fillna(0).sum()
lat = pd.to_numeric(df["latencia_s"], errors="coerce")

RESUMO = {
    "e1_rate_restrict": round(n_restrict_ok / n_restrict, 4) if n_restrict else None,
    "e1_rate_restrict_n": n_restrict_ok,
    "e1_rate_restrict_d": n_restrict,
    "e1_rate_overall": round(n_overall_ok / n_overall, 4) if n_overall else None,
    "e1_rate_overall_n": n_overall_ok,
    "e1_rate_overall_d": n_overall,
    "latencia_mediana_s": round(float(lat.median()), 2) if len(lat) else None,
    "latencia_media_s": round(float(lat.mean()), 2) if len(lat) else None,
    "chamadas_llm": int(df["chamadas_llm"].sum()) if len(df) else 0,
    "tokens_entrada": int(tok_in),
    "tokens_saida": int(tok_out),
    "custo_estimado_usd": round(tok_in * USD_IN + tok_out * USD_OUT, 6),
    "nota_custo": "estimativa indicativa flash-class; conferir tabela vigente do Gemini",
}

display(HTML(render_metrics_panel(RESUMO, n_restrict_ok, n_restrict, n_overall_ok, n_overall)))

print(json.dumps(RESUMO, indent=2, ensure_ascii=False))

out_json = STEP_DIR / "baseline_v1_resultados.json"
payload = {"run": RUN_INFO, "resumo": RESUMO, "registros": registros}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("salvo:", out_json)


e1_rate_restrictT01–T10 · baseline deve acertar,7/10 (70.0%)
e1_rate_overallT01–T14 · inclui gaps G,7/14 (50.0%)
Latência mediana,2.43 s
Latência média,2.41 s
Chamadas LLM,14
Tokens entrada / saída,"377,363 / 5,115"
Custo estimado (USD),$0.0398


{
  "e1_rate_restrict": 0.7,
  "e1_rate_restrict_n": 7,
  "e1_rate_restrict_d": 10,
  "e1_rate_overall": 0.5,
  "e1_rate_overall_n": 7,
  "e1_rate_overall_d": 14,
  "latencia_mediana_s": 2.43,
  "latencia_media_s": 2.41,
  "chamadas_llm": 14,
  "tokens_entrada": 377363,
  "tokens_saida": 5115,
  "custo_estimado_usd": 0.039782,
  "nota_custo": "estimativa indicativa flash-class; conferir tabela vigente do Gemini"
}
salvo: /home/anderson/Documents/Data-Science/LLM Courses/Unicamp-LLM-Agents/modules/04_projeto_pratico/step_01/baseline_v1_resultados.json


# 18. Análise crítica do baseline

Run registrado em `RUN_INFO` · `golden_revision` na seção 14.

**Leia as seções 16–17 após Run all:** a tabela compara baseline × gabarito; as notas `e1_rate_restrict` e `e1_rate_overall` vêm da seção 17. Nos casos com lista, **sucesso exige igualdade exata** com o gabarito pandas (mesmos SKUs, mesma ordem).

1. **O que atende.** Conferir na tabela os casos com `status_final=sucesso` no escopo **restrito** (T01–T10): filtro de categoria/marca, abstenções corretas, janela 7d, ordem do ranking.
2. **O que não atende.** Casos com `status_final=erro` no restrito: tipicamente `gabarito ≠ saída` (SKU ou ordem divergente) ou abstenção incorreta. Casos `erro*` (T11–T14) são gaps previstos do E1 (preço, claim).
3. **Erros típicos do stuffing.** Devolver ranking “quase certo” (mesma categoria, SKUs trocados ou ordem errada) conta como **erro** — não basta diversidade parcial ou filtro correto.
4. **Entradas difíceis.** Ambiguidade (T08/T09) e marca inexistente (T07) costumam ser pontos fortes do baseline; listas longas no prompt favorecem copiar subconjuntos do Top-5 sem reproduzir o gabarito.
5. **Modelo vs arquitetura.** Divergência de SKUs/ordem é limitação do modelo sobre tabela grande. Ausência de preço, estoque e claim é da **arquitetura** (não estão no contexto).
6. **Medida.** `e1_rate_restrict` isola T01–T10; `e1_rate_overall` inclui gaps G e deve subir nas evoluções incrementais.

Campeão do mês plantado: **Clash** (`6D2W9K`) tem vendas altas em 01–24/08 e baixas na janela 25–31/08.


# 19. Possíveis evoluções arquiteturais

| Peça | Limitação do E1 que atacaria | Custo | Nesta disciplina |
| :--- | :--- | :--- | :--- |
| **Tool text-to-SQL** | Agregar 1 240 linhas no prompt; erro de janela 7d | 1 tool + allowlist | **E2 (primeiro incremento)** |
| **Tools preço / estoque** | T12; ruptura | 2 adapters; o agente passa a ver fatos mutáveis | E2, depois do SQL |
| **Workflow LangGraph** | Etapas fixas: interpretar → consultar fatos → montar Top-5 | estado + `max_steps` | obrigatório no E2 |
| **RAG / fichas** | T11, T13, T14 (claims) | ingestão + groundedness | quando o SQL não resolver necessidade |
| **ReAct** | ordem das tools não for fixa | mais tokens; modos de falha de tool | só se o workflow linear falhar |
| **MCP** | preço/estoque reusados por ≥2 nós | servidor extra | não no E2 se tool local bastar |
| **Memória** | restrições acumuladas entre turnos | custo de contexto | **descartada agora** (sessão 1-shot); só com caso que falhe sem estado |
| **Auditor de fairness / multiagente** | concentração residual além do piso RF-07 (n_brands ≥ 2) | supervisor + especialista | E3, se o piso de duas marcas não bastar |
| **Planejamento** | query composta (orçamento + claim + lançamento) | mais chamadas | se o workflow E2 errar compostos |

Não usar peça por moda. Empate ou piora no eval é resultado válido.


# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final apresenta vantagens em relação ao baseline?**

Reexecutando o **mesmo** golden-set (`golden_revision`), o **mesmo** schema e as **mesmas** funções de verify, no **mesmo modelo** (`model_version`) e na mesma sessão, com o baseline stuffing intacto.

Evidência de vantagem (hipótese):

- `e1_rate_restrict` (T01–T10) **não piora**;
- `e1_rate_overall` (14 casos) **sobe** com preço, claim e demais peças;
- T12 passa a respeitar teto com preço **praticado** (tool), não `base_price`;
- T11/T13/T14 passam a acertar claim com citação da ficha (RAG), sem inventar;
- T05 continua a excluir o campeão do mês (SQL na janela 25–31/08, não stuffing);
- T01/T02/T04/T05 mantêm **n_brands ≥ 2** (RF-07); tools podem *facilitar* o mesmo requisito, não criá-lo do zero;
- latência, `llm_calls`, `tool_calls` e custo são **reportados** (não precisam ser menores).

Não promover SQL, RAG ou multiagente sem essa régua. Empate/piora numa dimensão entra no relatório com hipótese.


# 21. Conclusão

O RecFair ataca vitrine enviesada por popularidade e claims sem evidência. O **baseline V1 é parcial**: stuffing de catálogo + vendas diárias, Top-5 na janela **25–31/08/2026**, uma chamada Gemini (`model_version`), schema largo com preço/estoque/claim nulos.

A régua de 14 casos já contém o que o V1 deve acertar (categoria, abstenção, janela 7d, **diversidade mínima**) e o que deve falhar (preço, anticaspa, vegano, alta fixação). Copiar o Top-5 de popularidade em cabelos (5× Match) **reprova** T01. O próximo incremento justificado pelo tamanho da tabela é a **tool text-to-SQL** sobre as mesmas CSVs/`tb_*` no SQLite.


---

# Checklist antes da entrega

- [x] O problema está claramente definido.
- [x] O usuário-alvo foi identificado.
- [x] Escopo e não-objetivos estão explícitos.
- [x] Existem requisitos funcionais **verificáveis**.
- [x] Existem requisitos não funcionais.
- [x] Cada critério de sucesso diz **como** será medido.
- [x] O tipo de baseline foi classificado e justificado.
- [x] O baseline executa sem erros *(Run all neste notebook)*.
- [x] Existem pelo menos três casos de teste, cobrindo mais de um tipo (14 casos).
- [x] Modelo, temperatura e data da execução estão registrados (`model_version`).
- [x] Latência, tokens e número de chamadas foram registrados *(após Run all)*.
- [x] Os resultados estão na tabela e interpretados *(após Run all)*.
- [x] As limitações foram analisadas.
- [x] A pergunta obrigatória foi respondida.
- [x] O notebook foi executado do início ao fim e **salvo com as saídas**.
- [x] O notebook pode ser executado por outra pessoa (`generate_catalog.py` na mesma pasta).
- [x] Nenhuma chave de API foi incluída no notebook.
